**PRÁCTICA 5: ENTRENAMIENTO BASADO EN AUTOCARAS (PCA SOBRE PÍXELES + SVM)**

En este cuaderno se desarrolla un enfoque clásico basado en la técnica de **autocaras** (*Eigenfaces*), proyectando la intensidad directa de los píxeles en un subespacio de baja dimensión mediante **PCA** y clasificando las proyecciones mediante una **SVM RBF**.

A diferencia de HOG o LBP, este método modela la variabilidad global en intensidad. Para mitigar el coste computacional se emplean rostros a menor escala (32x32 píxeles, `TAM_PIX`), proyectando a 150 componentes principales con blanqueamiento (`whiten=True`), sirviendo como línea base de comparación directa respecto a los descriptores locales orientados a expresión.

**IMPORTS NECESARIOS Y DEPENDENCIAS**

Se importan las herramientas de descomposición en componentes principales, empaquetado de pipelines y clasificación de scikit-learn, junto con las constantes y funciones de preprocesado de `comun.py`.

In [1]:
import time
import joblib
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.svm import SVC
from comun import TAM_PIX, Extractor, cargar_dataset, informe, registrar_entrenamiento

**CONFIGURACIÓN DE RUTAS Y ARCHIVO DE SALIDA**

Se configuran los directorios de datos y se fija el archivo de salida en `autocaras_svm.joblib`.

In [2]:
RUTA_TRAIN = "dataset/train"
RUTA_TEST = "dataset/test"
SALIDA = "modelos/autocaras_svm.joblib"

**CARGA Y NORMALIZACIÓN DEL CONJUNTO DE DATOS (ESCALA REDUCIDA)**

Se cargan las imágenes redimensionándolas a una resolución adaptada de 32x32 píxeles (`TAM_PIX`), adecuada para aplanar los rostros en vectores de intensidad directa sin incurrir en una dimensionalidad desmesurada.

In [3]:
imgs_tr, ytr = cargar_dataset(RUTA_TRAIN, tam=TAM_PIX)
imgs_te, yte = cargar_dataset(RUTA_TEST, tam=TAM_PIX)

dataset/train: 30416 imágenes | por clase: {'angry': 3995, 'duckface': 2143, 'fear': 4097, 'happy': 7215, 'neutral': 4965, 'sad': 4830, 'surprise': 3171}
dataset/test: 7558 imágenes | por clase: {'angry': 958, 'duckface': 491, 'fear': 1024, 'happy': 1774, 'neutral': 1233, 'sad': 1247, 'surprise': 831}


**EXTRACCIÓN DEL VECTOR DE INTENSIDAD DE PÍXELES**

Utilizando el método `pixeles` de `Extractor`, se transforma cada imagen bidimensional en un vector unidimensional normalizado de 1024 valores de intensidad.

In [4]:
extractor = Extractor("pixeles")
Xtr = extractor.transform(imgs_tr)
Xte = extractor.transform(imgs_te)
print("Forma de X:", Xtr.shape)

Forma de X: (30416, 1024)


**DEFINICIÓN DEL PIPELINE DE AUTOCARAS (PCA CON BLANQUEAMIENTO + SVM)**

Se diseña la arquitectura encadenando un `PCA` configurado con 150 componentes principales y blanqueamiento espectral (`whiten=True`, que normaliza la varianza de cada autovector), seguido de un clasificador `SVC` con kernel RBF y ponderación de clases balanceada.

In [5]:
clasificador = make_pipeline(
    PCA(n_components=150, whiten=True, random_state=42),
    SVC(kernel="rbf", C=5, class_weight="balanced"),
)

**ENTRENAMIENTO DEL MODELO DE AUTOCARAS**

Se ajustan simultáneamente el espacio de proyección de autocaras y el clasificador SVM con los vectores aplanados de entrenamiento, registrando el tiempo transcurrido.

In [6]:
t0 = time.time()
clasificador.fit(Xtr, ytr)
t_entreno = time.time() - t0

**EVALUACIÓN CUANTITATIVA SOBRE EL CONJUNTO DE TEST**

Se proyectan las imágenes del conjunto de test sobre los autovectores aprendidos, se clasifican y se generan las estadísticas de precisión, recall y F1-score por clase.

In [7]:
t0 = time.time()
pred = clasificador.predict(Xte)
t_pred = time.time() - t0
metricas = informe(yte, pred, t_entreno, t_pred)

              precision    recall  f1-score   support

       angry       0.37      0.36      0.36       958
    duckface       0.75      0.76      0.75       491
        fear       0.41      0.38      0.39      1024
       happy       0.60      0.69      0.64      1774
     neutral       0.44      0.48      0.46      1233
         sad       0.37      0.34      0.35      1247
    surprise       0.75      0.60      0.67       831

    accuracy                           0.51      7558
   macro avg       0.53      0.52      0.52      7558
weighted avg       0.51      0.51      0.51      7558

Matriz de confusión (filas = real, columnas = predicho):
[[ 343   14  105  174  142  157   23]
 [  14  373   14   33   20   36    1]
 [ 132   21  386  136  133  156   60]
 [ 111   33   76 1228  161  134   31]
 [ 119   20   89  194  586  201   24]
 [ 160   33  173  209  219  427   26]
 [  52    4   90   71   63   50  501]]

Accuracy test: 0.5086 | F1 macro: 0.5196
Tiempo de entrenamiento: 1357.5 s
Inf

**SERIALIZACIÓN Y GUARDADO DEL PIPELINE COMPLETO**

Se ensambla el extractor de píxeles junto con el clasificador entrenado en un único objeto `Pipeline` de scikit-learn y se persiste con `joblib`.

In [8]:
modelo = Pipeline([("extractor", extractor), ("clasificador", clasificador)])
joblib.dump(modelo, SALIDA)
print("Modelo guardado en", SALIDA)
registrar_entrenamiento(SALIDA, t_entreno, ms_imagen=1000 * t_pred / len(yte), **metricas)

Modelo guardado en autocaras_svm.joblib
Registrado en informe_entrenamiento\registro_modelos.json
